# 5 · Meshes & density

`trimesh` draws an unstructured triangular mesh (a UGRID mesh, or a Delaunay triangulation of points).
`hexbin` and `kde` summarise a point cloud — hexagonal binning and a 2-D kernel-density surface. Density
plots want *many* points, so here we use a synthetic cloud of ~2 000 'sample' locations (clearly a stand-in
for GPS/sensor fixes) alongside the real gauges for the mesh.

**Setup** — Bokeh extension, the real gauges (for the mesh) and a synthetic point cloud (for density).

In [ ]:
from pathlib import Path

# Resolve the repo root so the bundled sample data is found whether this runs from
# docs/examples/interactive/ (mkdocs) or the repository root.
ROOT = Path.cwd()
while not (ROOT / "examples" / "data" / "LisbonElevation.tif").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA = ROOT / "examples" / "data"

import holoviews as hv
hv.extension("bokeh")            # the interactive tier renders through Bokeh

import numpy as np, geopandas as gpd
from pyramids.feature import FeatureCollection
from digitalearth.interactive import InteractiveMap

gauges = FeatureCollection.read_file(str(DATA / "rhine_gauges.geojson"))

rng = np.random.default_rng(0)                       # ~2k synthetic 'samples' over the basin
cloud = gpd.GeoDataFrame(
    {"v": rng.normal(10, 2, 2000)},
    geometry=gpd.points_from_xy(rng.uniform(6.0e5, 9.0e5, 2000), rng.uniform(6.2e6, 6.5e6, 2000)),
    crs="EPSG:3857",
)

### `trimesh` — an unstructured mesh
Triangulate the gauge locations (Delaunay) and colour the mesh by `discharge`. `rasterize=False` keeps the raw triangles; with many nodes it auto-rasterises via Datashader.

In [ ]:
m = InteractiveMap(crs=3857, title="gauge mesh (discharge)")
m.trimesh(gauges, value_column="discharge", rasterize=False, cmap="plasma")
m

### `hexbin` — hexagonal binning
Aggregate the cloud into hex tiles. `aggregator` picks the statistic (`count`, `mean`, …); with a `column` it aggregates that value, else point counts.

In [ ]:
m = InteractiveMap(crs=3857, title="point density (hexbin)")
m.hexbin(cloud, gridsize=25, aggregator="count")
m

### `kde` — a kernel-density surface
A smooth 2-D density of the same positions. `filled=True` shades the surface; `filled=False` draws density iso-lines.

In [ ]:
m = InteractiveMap(crs=3857, title="kernel density")
m.kde(cloud, filled=True, cmap="magma")
m